INITIAL SETUP

In [ ]:
import requests
import pandas as pd

1. SENSOR DATA PREPROCESSING

In [ ]:
url = f"https://api.thingspeak.com/channels/{CHANNEL_ID}/feeds.json?api_key={READ_API_KEY}&results=100"

response = requests.get(url)
data = response.json()

In [ ]:
feeds = data['feeds']
df = pd.DataFrame(feeds)
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.rename(columns={
    "field3": "Temperature",
    "field2": "Pulse(HR)",
    "field1": "spo2"
})
df

,created_at,entry_id,spo2,Pulse(HR),Temperature
0,2025-09-28 17:19:13+00:00,1,99,74,36.9
1,2025-09-28 17:19:33+00:00,2,99,76,36.9
2,2025-09-28 17:19:53+00:00,3,99,76,36.8
3,2025-09-28 17:20:13+00:00,4,98,77,36.9
4,2025-09-28 17:20:33+00:00,5,99,78,36.7
...,...,...,...,...,...
75,2025-09-28 17:44:21+00:00,76,96,90,36.6
76,2025-09-28 17:44:41+00:00,77,97,90,36.5
77,2025-09-28 17:45:02+00:00,78,98,89,36.5
78,2025-09-28 17:45:22+00:00,79,98,90,36.7


2. HUMAN VITAL SIGNS DATA PREPROSESSING

In [ ]:
pip install llama-index

INFO: pip is looking at multiple versions of llama-index-cli to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1

In [ ]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 65.2 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
import numpy as np

# 1. Load dataset
df = pd.read_csv("/content/human_vital_signs_dataset_2024.csv")

# 2. Convert rows into textual form
documents = [
    f"Patient ID: {row['Patient ID']}, Age: {row['Age']}, Gender: {row['Gender']}, "
    f"Heart Rate: {row['Heart Rate']}, Temp: {row['Body Temperature']:.2f}, "
    f"SpO2: {row['Oxygen Saturation']:.2f}, BP: {row['Systolic Blood Pressure']}/{row['Diastolic Blood Pressure']}, "
    f"Risk: {row['Risk Category']}"
    for _, row in df.iterrows()
]

# 3. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 4. Generate embeddings
embeddings = model.encode(documents, batch_size=64, show_progress_bar=True)

# 5. Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# 6. Store textual data in new dataset
df_with_text = df.copy()
df_with_text["Textual Data"] = documents
df_with_text.to_csv("/content/human_vital_signs_textual_dataset.csv", index=False)

# 7. Optionally save embeddings for reuse
np.save("/content/human_vital_signs_embeddings.npy", embeddings)

print("✅ Dataset with textual data saved as 'human_vital_signs_textual_dataset.csv'")
print("✅ Embeddings saved as 'human_vital_signs_embeddings.npy'")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3126 [00:00<?, ?it/s]

✅ Dataset with textual data saved as 'human_vital_signs_textual_dataset.csv'
✅ Embeddings saved as 'human_vital_signs_embeddings.npy'


3. USER INTERFACE

In [ ]:
!pip install -q google-generativeai gradio

import gradio as gr
import google.generativeai as genai
import pandas as pd
import numpy as np

# --- SETUP ---
genai.configure(api_key= api)
model = genai.GenerativeModel("gemini-2.0-flash")

# --- Load datasets ---
# df comes from ThingSpeak feeds
df = pd.DataFrame(feeds)
df['created_at'] = pd.to_datetime(df['created_at'])
# Rename columns consistently
df = df.rename(columns={
    "field1": "Temperature",
    "field2": "Heart Rate",
    "field3": "SpO2"
})

# Ensure numeric
df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce")
df["Heart Rate"] = pd.to_numeric(df["Heart Rate"], errors="coerce")
df["SpO2"] = pd.to_numeric(df["SpO2"], errors="coerce")

# Load textual RAG dataset
df_textual = pd.read_csv("/content/final_dataset.csv")  # last column 'Textual Data'

# --- Ask demographics before Gradio ---
print("Please provide your demographic details:")
user_age = int(input("Age: "))
user_gender = input("Gender (Male/Female/Other): ")
user_weight = float(input("Weight (kg): "))
user_height = float(input("Height (m): "))

profile = {
    "Age": user_age,
    "Gender": user_gender,
    "Weight": user_weight,
    "Height": user_height,
}

# --- Python-based analysis ---
def analyze_health(df):
    temps = df["Temperature"].dropna().tolist()
    pulses = df["Heart Rate"].dropna().tolist()
    spo2s = df["SpO2"].dropna().tolist()

    summary = f"""
✅ Python Health Data Analysis:
-------------------------------
• Number of readings: {len(df)}
"""
    if temps:
        summary += (
            f"• Avg Temp: {np.mean(temps):.2f} °C\n"
            f"• Min Temp: {min(temps):.2f} °C\n"
            f"• Max Temp: {max(temps):.2f} °C\n"
        )
    if pulses:
        summary += (
            f"• Avg Pulse: {np.mean(pulses):.2f} bpm\n"
            f"• Min Pulse: {min(pulses)} bpm\n"
            f"• Max Pulse: {max(pulses)} bpm\n"
        )
    if spo2s:
        summary += (
            f"• Avg SpO₂: {np.mean(spo2s):.2f}%\n"
            f"• Min SpO₂: {min(spo2s)}%\n"
            f"• Max SpO₂: {max(spo2s)}%\n"
        )

    return summary.strip()

python_analysis = analyze_health(df)

# --- Gemini initial summary ---
initial_prompt = (
    "You are a medical assistant.\n"
    "Analyze the following health sensor data (Temperature, Heart Rate, SpO₂):\n\n"
    f"{df.to_dict(orient='records')}\n\n"
    "Provide:\n"
    "- Average values\n"
    "- Detect abnormalities (fever > 37.5°C, low SpO₂ < 95, abnormal pulse)\n"
    "- Short doctor summary\n"
)
initial_response = model.generate_content(initial_prompt).text

# --- Rule-based diagnosis ---
def rule_based_diagnosis(temp, pulse, spo2):
    conditions = []
    if temp > 37.5:
        conditions.append("Possible fever (Temp > 37.5°C)")
    if temp < 35:
        conditions.append("Possible hypothermia (Temp < 35°C)")
    if spo2 < 95:
        conditions.append("Low SpO₂ (Hypoxemia risk)")
    if spo2 < 90:
        conditions.append("Severe hypoxemia – urgent checkup advised")
    if pulse > 100:
        conditions.append("Tachycardia (Pulse > 100 bpm)")
    elif pulse < 60:
        conditions.append("Bradycardia (Pulse < 60 bpm)")
    if temp > 37.5 and spo2 < 95:
        conditions.append("Fever + low SpO₂ → Possible respiratory infection")
    if pulse > 100 and spo2 < 95:
        conditions.append("High pulse + low SpO₂ → Possible hypoxemia or cardiac stress")
    if temp > 38 and pulse > 100:
        conditions.append("Fever + tachycardia → Possible sepsis or systemic infection")
    return conditions

# --- RAG-style analysis ---
def rag_analysis(profile):
    avg_temp = df["Temperature"].mean(skipna=True)
    avg_pulse = df["Heart Rate"].mean(skipna=True)
    avg_spo2 = df["SpO2"].mean(skipna=True)

    conditions = rule_based_diagnosis(avg_temp, avg_pulse, avg_spo2)

    # Use textual dataset to find common issues
    textual_data = df_textual["Textual Data"].tolist()

    prompt = (
        "You are a medical assistant.\n"
        f"User profile: {profile}\n"
        "Average vital signs from dataset:\n"
        f"Temperature: {avg_temp:.2f} °C, Pulse: {avg_pulse:.2f} bpm, SpO₂: {avg_spo2:.2f}%\n"
        f"Rule-based detected conditions: {conditions}\n\n"
        f"From past patient records (examples: {textual_data[:50]} ...), "
        "identify common issues with people of similar demographics and conditions.\n"
        "Provide a short, clear summary."
    )
    return model.generate_content(prompt).text

rag_output = rag_analysis(profile)

# --- Chat with Gemini ---
chat_history = []

def chat_with_bot(user_input, profile):
    response = model.generate_content(
        f"Given this health dataset:\n{df.to_dict(orient='records')}\n\n"
        f"Textual dataset:\n{df_textual.to_dict(orient='records')}\n\n"
        f"User profile: {profile}\n"
        f"User question: {user_input}"
    ).text
    chat_history.append((user_input, response))
    return chat_history

# --- Gradio UI ---
with gr.Blocks() as demo:
    gr.Markdown("## 🩺 Health Sensor Assistant (Gemini + Python + RAG)")

    gr.Markdown("### 👤 User Profile")
    gr.Textbox(value=str(profile), label="Demographics", interactive=False)

    gr.Markdown("### 🧠 Gemini’s Automatic Medical Summary")
    gr.Textbox(value=initial_response, label="Gemini Insights", lines=8, interactive=False)

    gr.Markdown("### 🧮 Local Python Analysis")
    gr.Textbox(value=python_analysis, label="Python-Based Summary", lines=10, interactive=False)

    gr.Markdown("### 🔍 Combined RAG Analysis")
    gr.Textbox(value=rag_output, label="RAG-Style Insights", lines=8, interactive=False)

    gr.Markdown("### 💬 Ask Questions")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask a question about your health data...")
    send_btn = gr.Button("Send")

    def user_send(user_msg):
        return chat_with_bot(user_msg, profile), ""

    send_btn.click(user_send, inputs=[msg], outputs=[chatbot, msg])

demo.launch(share=True)


Please provide your demographic details:
Age: 53
Gender (Male/Female/Other): Female
Weight (kg): 65
Height (m): 155


/tmp/ipython-input-605830751.py:169: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bebc8b497f9e8328bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
